# Kernel provenance with torch.compile
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week5/Provenance_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 5, Module 3: Observability across the stack**

When torch.compile fuses several operations into one GPU kernel, the kernel name lists the fused ATen ops but not the lines of Python they came from. **Provenance tracking** records that link. In this notebook you compile a small attention block, generate a provenance report, and trace one fused kernel back to its source lines.

**What you will do**
1. Check the GPU and install tlparse.
2. Write a small attention block to a file.
3. Compile it with logging and provenance turned on.
4. Turn the log into an HTML report with tlparse.
5. Open the report and read the provenance view.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order. The whole notebook takes about two minutes.

## 1. Check the environment
This cell confirms that a GPU is available and installs **tlparse**, the PyTorch tool that turns a compile log into an HTML report.

Provenance tracking needs PyTorch 2.8 or later. Note the versions printed here: results can differ between versions.

In [ ]:
# Install tlparse, the tool that reads torch.compile logs and writes an HTML report.
!pip -q install tlparse

import subprocess
import torch

# Provenance tracking needs PyTorch 2.8 or later.
print("PyTorch:", torch.__version__)

# The kernels in this demo are Triton GPU kernels, so a CUDA GPU is required.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("tlparse:", subprocess.run(["tlparse", "--version"], capture_output=True, text=True).stdout.strip())

## 2. Write the model to a file
The model is a small attention block: a layer norm, a projection to q, k and v, scaled scores with a causal mask, a softmax, and an output projection with GELU and a residual.

It is written to a file so that the compiler can record **file names and line numbers**. The comment at the end of each line in `forward` names the operation. Keep these line numbers in mind: the report points back to them.

| Line | Operation |
|---|---|
| 20 | layer norm |
| 21 | q, k, v projection |
| 23 to 25 | reshape q, k, v into heads |
| 26 | attention scores and scale |
| 27 | additive mask |
| 28 | softmax |
| 29 | weighted sum of values |
| 30 | output projection, GELU, residual |

In [ ]:
%%writefile /content/provenance_demo.py
# Week 5, Module 3: provenance tracking demo.
# Run with:
#   TORCH_TRACE=/content/trace INDUCTOR_PROVENANCE=1 python provenance_demo.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    # Small attention-style block. Inductor fuses the scale, mask and softmax
    # into one kernel, so that kernel maps back to several source lines.
    def __init__(self, d=256, h=4):
        super().__init__()
        self.h = h
        self.norm = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x, mask):
        y = self.norm(x)                                    # layer norm
        q, k, v = self.qkv(y).chunk(3, dim=-1)               # one matmul, split
        B, T, D = q.shape
        q = q.view(B, T, self.h, D // self.h).transpose(1, 2)
        k = k.view(B, T, self.h, D // self.h).transpose(1, 2)
        v = v.view(B, T, self.h, D // self.h).transpose(1, 2)
        s = (q @ k.transpose(-1, -2)) * (D // self.h) ** -0.5  # scores and scale
        s = s + mask                                         # additive mask
        p = torch.softmax(s, dim=-1)                         # softmax
        o = (p @ v).transpose(1, 2).reshape(B, T, D)
        return x + F.gelu(self.proj(o))                      # projection, GELU, residual


def main():
    torch.manual_seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = Block().to(dev).eval()
    x = torch.randn(2, 128, 256, device=dev)
    # Causal mask: -1e4 above the diagonal. A finite value, so it encodes safely in fp16.
    mask = torch.triu(torch.full((128, 128), -1e4, device=dev), diagonal=1)

    compiled = torch.compile(model)
    with torch.no_grad():
        out = compiled(x, mask)   # compiled run: this is what gets logged
        ref = model(x, mask)      # eager run, as a baseline
    print("torch", torch.__version__, "| device", dev)
    # A small difference here is rounding. The compiled and eager runs agree.
    print("max abs diff vs eager:", (out - ref).abs().max().item())


if __name__ == "__main__":
    main()

**Reading the output**
- `torch 2.x | device cuda`: the PyTorch version and the device. Record the version.
- `max abs diff vs eager`: about 5e-7. That is float32 rounding. The compiled and eager runs agree.
- Two warnings may appear. `dynamo_pgo force disabled` comes from turning off the caches. `Not enough SMs to use max_autotune_gemm` means the T4 is too small for one autotuning mode. Neither affects this demo.
- The trace folder holds one log file, about 300 KB. Everything in the next step is built from it.

## 3. Compile with logging and provenance turned on
Three environment variables control what the compiler records:

| Variable | What it does |
|---|---|
| `TORCH_TRACE=/content/trace` | Writes a structured log of the whole compile to this folder. |
| `INDUCTOR_PROVENANCE=1` | Adds the mapping from each generated kernel back to the source ops and lines. |
| `TORCHINDUCTOR_FORCE_DISABLE_CACHES=1` | Turns off the compile cache. With a warm cache, nothing recompiles and no artifacts are written. |

The variables are set on the same line as the command, so they apply only to this run. The compile takes 30 to 60 seconds.

In [ ]:
# Start from a clean folder so the report shows only this run.
!rm -rf /content/trace /content/tl_out

# Compile and run the model with logging and provenance turned on.
!TORCH_TRACE=/content/trace INDUCTOR_PROVENANCE=1 TORCHINDUCTOR_FORCE_DISABLE_CACHES=1 python /content/provenance_demo.py

# The trace folder now holds one log file per process.
!ls -lh /content/trace

## 4. Build the HTML report with tlparse
tlparse reads the log and writes an HTML report to `/content/tl_out`. The `--inductor-provenance` option adds the provenance view.

Pass tlparse the **log file**, not the folder.

In [ ]:
import glob

# Pick the newest log file in the trace folder.
log = sorted(glob.glob('/content/trace/*.log'))[-1]
print("Log file:", log)

# Write the report. --overwrite replaces any earlier report in the same folder.
# The progress bar is long, so show only the last lines of tlparse's output.
!tlparse {log} --inductor-provenance -o /content/tl_out --overwrite 2>&1 | tail -n 3

# index.html is the report's front page. The provenance page is listed with it.
!ls /content/tl_out | head -20

**What tlparse wrote**
Open the folder icon on the left and expand `tl_out`. You do not need to open these files one by one. The report index links to all of them. They fall into four groups:

| Group | Files | What they are |
|---|---|---|
| Report pages | `index.html`, `provenance_tracking_-_0_0_0.html`, `failures_and_restarts.html` | The pages you open in the browser. |
| One folder per compile | `-_0_0_0/` | Frame 0, compile 0. Its files are the compiler's outputs in pipeline order, numbered 1 to 19. |
| Inside that folder | `dynamo_output_graph`, `aot_inference_graph`, `inductor_post_grad_graph`, `inductor_output_code_...` | The captured graph, the AOTAutograd graph, the graph after Inductor's passes, and the generated Triton code. |
| | `inductor_provenance_tracking_...` | The kernel-to-source mapping that `INDUCTOR_PROVENANCE=1` added. |
| Raw data | `raw.jsonl`, `compile_directory.json`, `chromium_events.json`, `payloads/` | The parsed log and a compile-time trace for Perfetto. Not needed here. |

These are the same per-stage artifacts that `TORCH_COMPILE_DEBUG=1` writes. tlparse collects them in one place and links them.

## 5. Open the report
This cell serves the report from this Colab session and prints two links. Click each one to open it in a new browser tab. The links work only while this runtime is connected.

- **Report index:** a summary of the compile. Under *Build products*, the entry `[0/0]` means frame 0, compile 0: the model compiled once, with no graph breaks and no recompiles. The files listed under it are what the compiler wrote, in pipeline order: the Dynamo graph, the AOTAutograd graph, the post-grad graph, the generated code, and the provenance data.
- **Provenance page:** three linked panes, described in the next section.

If a link shows an error, run the cell again. The server stops when the runtime restarts.

In [ ]:
import subprocess, time

# Serve the report folder on port 8000 inside the Colab runtime.
subprocess.Popen(["python", "-m", "http.server", "8000", "--directory", "/content/tl_out"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# Ask Colab for the public address of port 8000, then print one link per page.
# Click a link to open that page in a new browser tab.
from google.colab.output import eval_js
base = eval_js("google.colab.kernel.proxyPort(8000)")
print("Report index:    ", base + "index.html")
print("Provenance view: ", base + "provenance_tracking_-_0_0_0.html")

## 6. Read the provenance view
The provenance page has three panes. Each one is a stage of the compile.

| Pane | What it shows | Compiler stage |
|---|---|---|
| Left | The graph Dynamo captured. Each block starts with `# File: provenance_demo.py:N`, the source line. | Graph capture |
| Middle | The graph after AOTAutograd and decompositions. High-level ops are split into ATen and prims ops. | Autograd, decomposition |
| Right | The generated code. The `call()` function launches each kernel in order. | Fusion, code generation |

**Try this**
1. In the right pane, find the kernel whose name starts with `triton_per_fused__softmax_add_mul`. Click it. The left pane highlights three source lines: 26 (scores and scale), 27 (mask) and 28 (softmax). Fusion merged three lines of Python into one kernel.
2. In the middle pane, find the block under **No stacktrace found**. A decomposition inserted that op, so it has no source line. This is expected.
3. Click the kernel ending in `_view_1`. It maps to a single line, 23: the reshape of q into heads.
4. Find `extern_kernels.mm` and `extern_kernels.bmm`. The matrix multiplies are sent to cuBLAS rather than generated as Triton kernels, so they are not fused with the ops around them.

Line numbers in the right pane depend on the PyTorch version. Find the kernels by name.

## 7. Optional: keep a copy of the report
Downloads the report as a zip. To open it later without Colab, unzip it, run `python3 -m http.server 8000` inside the `tl_out` folder, and open http://localhost:8000.

In [ ]:
# Zip the report folder and download it.
!cd /content && zip -qr tl_out.zip tl_out
from google.colab import files
files.download('/content/tl_out.zip')

## Summary
- A fused kernel's name lists its ATen ops, not its source lines.
- `TORCH_TRACE` and `INDUCTOR_PROVENANCE=1` record the compile and the link from each kernel to its source.
- `tlparse --inductor-provenance` turns the log into a report with three linked panes: captured graph, post-grad graph, generated code.
- An op with no stack trace was inserted by the compiler. That is expected.

Reference: PyTorch documentation, Inductor provenance tracking, and the torch.compile troubleshooting guide.